In [ ]:
import torchvision.models as visionmodels
import torch.nn as nn
import torch.optim as optim
import torch

class Emolens(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = visionmodels.efficientnet_b4(weights=visionmodels.EfficientNet_B4_Weights.DEFAULT)

        for parameter in self.model.parameters():
            parameter.requires_grad = False

        for parameter in self.model.features[-1].parameters():
            parameter.requires_grad = True
        for parameter in self.model.features[-2].parameters():
            parameter.requires_grad = True
        for parameter in self.model.features[-3].parameters():
            parameter.requires_grad = True
        for parameter in self.model.features[-4].parameters():
            parameter.requires_grad = True

        self.model.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(1792, 8))

    def forward(self, x):
        x = self.model(x)

        return x

In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('mv kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')
os.system('kaggle datasets download -d mstjebashazida/affectnet -p /content/ --unzip')

Mounted at /content/drive


0

In [ ]:
!mv 'archive (3)' data
!ls data

data  drive  sample_data


In [8]:
!mv 'data/Test/Anger' 'data/Test/anger'
!mv 'data/Test/Contempt' 'data/Test/contempt'

In [9]:
train_Data = "data/Train"
Test_Data = "data/Test"
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_dataset = datasets.ImageFolder(root=train_Data, transform=train_transform)
test_dataset = datasets.ImageFolder(root=Test_Data, transform=test_transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
model = Emolens()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
model_parameters = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(model_parameters, lr=0.00005)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',       # monitor loss -- lower is better
    patience=2,       # wait 2 epochs before reducing
    factor=0.5        # reduce lr by half when triggered
)

In [11]:
print(train_dataset.class_to_idx)
print(test_dataset.class_to_idx)

{'anger': 0, 'contempt': 1, 'disgust': 2, 'fear': 3, 'happy': 4, 'neutral': 5, 'sad': 6, 'surprise': 7}
{'anger': 0, 'contempt': 1, 'disgust': 2, 'fear': 3, 'happy': 4, 'neutral': 5, 'sad': 6, 'surprise': 7}


In [27]:
num_of_epochs=10

for epoch in range(num_of_epochs):
    sum_of_train_losses = 0
    sum_of_test_losses = 0
    total_correct = 0
    total_images = 0

    model.train()
    for images, labels in train_dataloader:
        images = images.to(device)
        labels = labels.to(device)
        
        y_pred = model(images)
        loss = criterion(y_pred, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_of_train_losses += loss.item()

    model.eval()
    with torch.no_grad():
        for images, labels in test_dataloader:
            images = images.to(device)
            labels = labels.to(device)
            
            y_pred = model(images)
            loss = criterion(y_pred, labels)

            y_pred = torch.argmax(y_pred, dim=1)
            total_correct += (y_pred == labels).sum().item()
            total_images += labels.size(0)

            sum_of_test_losses += loss.item()

    accuracy = (total_correct / total_images) * 100
    print(f'Epoch {epoch+1}: Train Loss: {sum_of_train_losses / len(train_dataloader): .3f} | Test Loss: {sum_of_test_losses / len(test_dataloader): .3f} | Accuracy: {accuracy: .2f}%')

Epoch 1: Train Loss:  1.748 | Test Loss:  1.519 | Accuracy:  43.26%
Epoch 2: Train Loss:  1.261 | Test Loss:  1.443 | Accuracy:  51.96%
Epoch 3: Train Loss:  1.127 | Test Loss:  1.356 | Accuracy:  55.41%
Epoch 4: Train Loss:  1.055 | Test Loss:  1.384 | Accuracy:  57.56%
Epoch 5: Train Loss:  0.995 | Test Loss:  1.346 | Accuracy:  59.48%
Epoch 6: Train Loss:  0.946 | Test Loss:  1.302 | Accuracy:  60.37%
Epoch 7: Train Loss:  0.901 | Test Loss:  1.290 | Accuracy:  61.53%
Epoch 8: Train Loss:  0.857 | Test Loss:  1.319 | Accuracy:  62.16%
Epoch 9: Train Loss:  0.820 | Test Loss:  1.288 | Accuracy:  63.05%
Epoch 10: Train Loss:  0.787 | Test Loss:  1.357 | Accuracy:  62.94%
